# INFO802 Assessment 3 – Lightweight vs complex phishing detection
**RQ2:** How does the detection performance of simple models (Naïve Bayes, Logistic Regression) compare with a transformer model (DistilBERT) when computing cost is also considered?

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.
Total time is usually 30–60 minutes. Results are saved to `results.xlsx` and two chart images.

## 1. Install and import

In [ ]:
!pip -q install kagglehub openpyxl
import os, glob, time, random, zipfile, tracemalloc, shutil, platform, joblib
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SEED = 42
SAMPLE_SIZE = 20000      # same stratified sample for every model
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. Record the hardware (evidence for the cost results)

In [ ]:
print("Python:", platform.python_version())
print("CPU:", platform.processor() or platform.machine(), "| cores:", os.cpu_count())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None - switch runtime to T4 GPU")

## 3. Load the Kaggle Phishing Email Dataset (82,486 emails)

In [ ]:
import kagglehub
kpath = kagglehub.dataset_download("naserabdullahalam/phishing-email-dataset")
print("Downloaded to:", kpath); print(os.listdir(kpath))

# The combined file is phishing_email.csv. If the name differs, pick the largest CSV.
csvs = glob.glob(os.path.join(kpath, "**", "*.csv"), recursive=True)
main_csv = next((c for c in csvs if os.path.basename(c).lower() == "phishing_email.csv"), max(csvs, key=os.path.getsize))
raw = pd.read_csv(main_csv)
print(main_csv, raw.shape); print(raw.columns.tolist())

text_col = next(c for c in ["text_combined", "body", "text", "Email Text"] if c in raw.columns)
label_col = next(c for c in ["label", "Label", "Email Type"] if c in raw.columns)
df = raw[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"})
if df["label"].dtype == object:   # e.g. 'Phishing Email' / 'Safe Email'
    df["label"] = df["label"].str.lower().str.contains("phish|spam").astype(int)

before = len(df)
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0].drop_duplicates(subset="text").reset_index(drop=True)
print(f"Rows before cleaning: {before:,} | after removing blanks and duplicates: {len(df):,}")
print(df["label"].value_counts().rename({1: "phishing/spam", 0: "legitimate"}))

## 4. Load the AI-generated phishing corpus (Eze & Shamir, 2024)
Source: https://people.cs.ksu.edu/~lshamir/data/ai_phishing/
If the download fails: open the link in your browser, download the files (or zip), then in Colab click the folder icon, right-click the `ai_phishing` folder → Upload, and run this cell again.

In [ ]:
import subprocess, tarfile
URL = "https://people.cs.ksu.edu/~lshamir/data/ai_phishing/"

# Start clean so an earlier failed attempt doesn't block the download
if not any(f.endswith(".txt") for f in glob.glob("ai_phishing/**/*", recursive=True)):
    shutil.rmtree("ai_phishing", ignore_errors=True)
    os.makedirs("ai_phishing", exist_ok=True)
    subprocess.run(["wget", "-q", "-r", "-np", "-nH", "--cut-dirs=3", "-e", "robots=off",
                    "--no-check-certificate", "-R", "index.html*", "-P", "ai_phishing", URL])

# Unpack any archives
for f in glob.glob("ai_phishing/**/*", recursive=True):
    try:
        if f.endswith(".zip"): zipfile.ZipFile(f).extractall(os.path.dirname(f))
        elif f.endswith((".tar", ".tar.gz", ".tgz")): tarfile.open(f).extractall(os.path.dirname(f))
    except Exception as e: print("Could not unpack", f, e)

# Show what was downloaded (helps if something goes wrong)
files_found = [f for f in glob.glob("ai_phishing/**/*", recursive=True) if os.path.isfile(f)]
print("Files found:", len(files_found))
for f in files_found[:10]: print(" ", f, os.path.getsize(f), "bytes")

ai_texts = []
for f in files_found:
    if f.endswith((".zip", ".tar", ".gz", ".tgz", ".html", ".gif", ".png")) or "robots" in f:
        continue
    with open(f, encoding="utf-8", errors="ignore") as fh:
        t = fh.read().strip()
    if len(t) > 50:
        ai_texts.append(t)
ai_texts = list(dict.fromkeys(ai_texts))   # remove duplicates
print("AI-generated phishing emails loaded:", len(ai_texts))   # expected about 865
assert len(ai_texts) > 100, "AI corpus not found - send a screenshot of the output above."
print(ai_texts[0][:400])

## 5. Stratified sample, 80/20 split, and the balanced AI test set

In [ ]:
sample, _ = train_test_split(df, train_size=SAMPLE_SIZE, stratify=df["label"], random_state=SEED)
train_df, test_df = train_test_split(sample, test_size=0.2, stratify=sample["label"], random_state=SEED)

legit_pool = test_df[test_df["label"] == 0]
n_ai = min(len(ai_texts), len(legit_pool))
ai_test = pd.concat([
    pd.DataFrame({"text": ai_texts[:n_ai], "label": 1}),
    legit_pool.sample(n_ai, random_state=SEED)[["text", "label"]],
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Train: {len(train_df):,} | Test: {len(test_df):,} | Balanced AI test: {len(ai_test):,} ({n_ai} AI phishing + {n_ai} legitimate)")
X_tr, y_tr = train_df["text"].tolist(), train_df["label"].values
X_te, y_te = test_df["text"].tolist(), test_df["label"].values
X_ai, y_ai = ai_test["text"].tolist(), ai_test["label"].values

## 6. Helper functions

In [ ]:
def scores(y, p):
    return {"Accuracy": accuracy_score(y, p), "Precision": precision_score(y, p, zero_division=0),
            "Recall": recall_score(y, p, zero_division=0), "F1": f1_score(y, p, zero_division=0)}

def folder_mb(path):
    return sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(path) for f in fs) / 1e6

perf_rows, cost_rows, cms = [], [], {}

## 7. Lightweight models: Naïve Bayes and Logistic Regression (TF-IDF features)

In [ ]:
def make(name):
    tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
    clf = MultinomialNB() if name == "Naïve Bayes" else LogisticRegression(max_iter=1000)
    return Pipeline([("tfidf", tfidf), ("clf", clf)])

for name in ["Naïve Bayes", "Logistic Regression"]:
    train_times, peaks = [], []
    for run in range(3):                       # timing averaged over 3 runs
        model = make(name)
        tracemalloc.start(); t0 = time.perf_counter()
        model.fit(X_tr, y_tr)
        train_times.append(time.perf_counter() - t0)
        peaks.append(tracemalloc.get_traced_memory()[1] / 1e6); tracemalloc.stop()

    t0 = time.perf_counter(); pred = model.predict(X_te); det = time.perf_counter() - t0
    pred_ai = model.predict(X_ai)
    joblib.dump(model, f"{name}.joblib")

    s, sa = scores(y_te, pred), scores(y_ai, pred_ai)
    perf_rows.append({"Model": name, **s, "AI-set accuracy": sa["Accuracy"], "AI-set recall": sa["Recall"], "AI-set F1": sa["F1"]})
    cost_rows.append({"Model": name, "Training time (s)": np.mean(train_times),
                      "Detection time per 1,000 emails (s)": det / len(X_te) * 1000,
                      "Peak memory (MB)": np.mean(peaks), "Memory type": "RAM (training)",
                      "Model size (MB)": os.path.getsize(f"{name}.joblib") / 1e6, "Hardware": "CPU"})
    cms[name] = confusion_matrix(y_te, pred)
    print(name, {k: round(v, 4) for k, v in s.items()}, "| AI-set recall:", round(sa["Recall"], 4))

## 8. Complex model: DistilBERT fine-tuned for 2 epochs (needs the GPU)

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)
enc = lambda b: tok(b["text"], truncation=True, max_length=256, padding="max_length")
ds_tr = Dataset.from_dict({"text": X_tr, "label": y_tr.tolist()}).map(enc, batched=True)
ds_te = Dataset.from_dict({"text": X_te, "label": y_te.tolist()}).map(enc, batched=True)
ds_ai = Dataset.from_dict({"text": X_ai, "label": y_ai.tolist()}).map(enc, batched=True)

bert = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args = TrainingArguments(output_dir="bert_out", num_train_epochs=2, per_device_train_batch_size=16,
                         per_device_eval_batch_size=64, learning_rate=2e-5, weight_decay=0.01,
                         fp16=torch.cuda.is_available(), save_strategy="no", logging_steps=200,
                         report_to="none", seed=SEED)
trainer = Trainer(model=bert, args=args, train_dataset=ds_tr)

if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter(); trainer.train(); bert_train = time.perf_counter() - t0
gpu_peak = torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else float("nan")

t0 = time.perf_counter(); pred = trainer.predict(ds_te).predictions.argmax(-1); det = time.perf_counter() - t0
pred_ai = trainer.predict(ds_ai).predictions.argmax(-1)
trainer.save_model("distilbert_model")

s, sa = scores(y_te, pred), scores(y_ai, pred_ai)
perf_rows.append({"Model": "DistilBERT", **s, "AI-set accuracy": sa["Accuracy"], "AI-set recall": sa["Recall"], "AI-set F1": sa["F1"]})
cost_rows.append({"Model": "DistilBERT", "Training time (s)": bert_train,
                  "Detection time per 1,000 emails (s)": det / len(X_te) * 1000,
                  "Peak memory (MB)": gpu_peak, "Memory type": "GPU (training)",
                  "Model size (MB)": folder_mb("distilbert_model"),
                  "Hardware": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})
cms["DistilBERT"] = confusion_matrix(y_te, pred)
print("DistilBERT", {k: round(v, 4) for k, v in s.items()}, "| AI-set recall:", round(sa["Recall"], 4))

## 9. Results tables, charts and Excel export

In [ ]:
perf = pd.DataFrame(perf_rows)
cost = pd.DataFrame(cost_rows)
pct = ["Accuracy", "Precision", "Recall", "F1", "AI-set accuracy", "AI-set recall", "AI-set F1"]
perf_pct = perf.copy(); perf_pct[pct] = (perf_pct[pct] * 100).round(1)
display(perf_pct); display(cost.round(3))

best_light = perf.iloc[:2]["F1"].max(); bert_f1 = perf.iloc[2]["F1"]
gap = (bert_f1 - best_light) * 100
print(f"F1 gap (DistilBERT - best lightweight): {gap:.2f} percentage points")
print("Decision rule (within 2 points):", "COMPARABLE" if gap <= 2 else "NOT comparable")

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(perf)); w = 0.38
ax.bar(x - w/2, perf_pct["F1"], w, label="Test set F1")
ax.bar(x + w/2, perf_pct["AI-set recall"], w, label="AI-set recall")
ax.set_xticks(x, perf["Model"]); ax.set_ylabel("%"); ax.set_ylim(0, 105); ax.legend()
ax.set_title("Detection performance by model"); plt.tight_layout(); plt.savefig("chart_performance.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(cost["Model"], cost["Training time (s)"]); ax.set_yscale("log")
ax.set_ylabel("Seconds (log scale)"); ax.set_title("Training time by model")
plt.tight_layout(); plt.savefig("chart_training_time.png", dpi=200); plt.show()

with pd.ExcelWriter("results.xlsx") as xw:
    perf_pct.to_excel(xw, sheet_name="Performance", index=False)
    cost.round(3).to_excel(xw, sheet_name="Cost", index=False)
    for n, cm in cms.items():
        pd.DataFrame(cm, index=["Actual legit", "Actual phishing"], columns=["Pred legit", "Pred phishing"]).to_excel(xw, sheet_name=n[:31])

from google.colab import files
for f in ["results.xlsx", "chart_performance.png", "chart_training_time.png"]:
    files.download(f)